# 07d — Train & Evaluate (70:30 train/test branch)

Fourth parallel branch alongside `07_train_eval.ipynb` (spatial k-fold,
formal baseline), `07b_train_eval_bootstrap.ipynb` (bootstrap resampling),
and `07c_train_eval_random_repeats.ipynb` (60/20/20 random repeats).

**Why this branch exists:** tests whether matching Lei et al. 2024's
reported 70:30 split ratio changes the picture, while keeping model
capacity fixed at the 07 baseline (`hidden_dim=64`, not 07b/c's 128) so
the split scheme is the one variable actually being isolated. Uses
REPEATED 70:30 TRAIN/TEST re-splits (`train.run_scenario_train_test_repeats`):
each repeat holds out a genuine 30% test split, with an internal val
carved OUT OF THE 70% TRAIN PORTION ONLY (never from test) purely for
early stopping + threshold fitting.

**Important difference from Lei et al.'s own script:** their notebook
(`gnn_building_prediction.ipynb`) uses the 30% test split itself for
early-stopping / best-checkpoint selection, i.e. test leaks into model
selection. This branch does NOT reproduce that — test here never
influences anything upstream of the final metrics. So this branch's
"test" numbers are directly comparable to the paper's split *ratio*, but
are a stricter (uncontaminated) estimate, not the same statistical object.

Threshold is FIXED at 0.5 (`configs/eval_70_30.yaml`), PR-AUC is the
primary selection/reporting metric, and accuracy (plus auroc/f1/precision/
recall) is reported every repeat via `evaluate.compute_metrics` regardless.

Descriptive aggregates only (mean +/- std across repeats) — like 07b/07c,
this doesn't feed evaluate.py's paired k-fold significance tests (see
`run_scenario`'s docstring for why).

In [ ]:
REPO_URL = "https://github.com/AditPradana36/crash-dualgraph.git"
REPO_DIR = "/content/crash-dualgraph"

import os
if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    !cd {REPO_DIR} && git pull

import sys
sys.path.append(f"{REPO_DIR}/src")

from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# TEMP: install locally patched train.py / models.py / plot_history.py
# until pushed to GitHub. Skip this cell once the repo itself is updated.
from google.colab import files
import shutil

print("Upload train.py, models.py, and plot_history.py:")
uploaded = files.upload()
for fname in uploaded:
    shutil.move(fname, f"{REPO_DIR}/src/{fname}")
print("Patched files installed:", list(uploaded.keys()))

In [ ]:
!pip install -q torch_geometric xgboost scikit-learn scipy pyyaml pandas tqdm

In [ ]:
import yaml
from pathlib import Path
import torch

with open(f"{REPO_DIR}/configs/paths.yaml") as f:
    paths_cfg = yaml.safe_load(f)
with open(f"{REPO_DIR}/configs/eval_70_30.yaml") as f:
    eval_cfg = yaml.safe_load(f)
with open(f"{REPO_DIR}/configs/model_70_30.yaml") as f:
    model_cfg = yaml.safe_load(f)

PROCESSED_DIR = Path(paths_cfg["processed_dir"])
OUTPUTS_DIR = Path(paths_cfg["outputs_dir"])
# separate checkpoint/metrics dirs from 07/07b/07c -- keeps this branch's
# results from colliding with any of the other three
CHECKPOINT_DIR = OUTPUTS_DIR / "checkpoints_70_30"
METRICS_DIR = OUTPUTS_DIR / "metrics_70_30"
for d in [CHECKPOINT_DIR, METRICS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

device = "cuda" if torch.cuda.is_available() else "cpu"
config = {"batch_size": eval_cfg.get("batch_size") or 128,
          "epoch_cap": eval_cfg.get("epoch_cap") or 350,
          "warmup_epochs": eval_cfg.get("warmup_epochs") or 75,
          "patience": eval_cfg.get("patience") or 30,
          "lr_patience": eval_cfg.get("lr_patience") or 5,
          "lr": eval_cfg.get("lr") or 5e-3,
          "weight_decay": eval_cfg.get("weight_decay") or 1e-4,
          "fusion_dim": model_cfg.get("fusion_dim") or 64,
          # 70:30 outer split; internal_val_frac carves val out of the
          # 70% TRAIN portion only -- test_frac is the true held-out 30%,
          # never touched by run_scenario_train_test_repeats's internal
          # early-stopping/threshold-fitting split.
          "test_frac": eval_cfg.get("test_frac") or 0.30,
          "internal_val_frac": eval_cfg.get("internal_val_frac") or 0.15,
          # stratified split: preserves (or gently nudges, via
          # target_pos_frac) the positive rate per split instead of a
          # plain shuffle drifting it by chance -- see
          # train._stratified_split's docstring
          "label_col": eval_cfg.get("label_col") or "label",
          "target_pos_frac": eval_cfg.get("target_pos_frac"),
          # fixed at 0.5 per this branch's design (not learned) --
          # explicit here rather than left to train_one_fold's default
          "threshold_method": eval_cfg.get("threshold_method") or "fixed",
          "threshold": eval_cfg.get("threshold", 0.5),
          "num_workers": eval_cfg.get("num_workers") or 0,
          "use_amp": eval_cfg.get("use_amp", True)}
N_REPEATS = eval_cfg.get("n_repeats") or 10
print(f"Device: {device} | n_repeats: {N_REPEATS}")
print(f"Split: 70%/30% (train_full/test), with an internal "
      f"{config['internal_val_frac']:.0%} of the 70% carved out as val "
      f"for early stopping only -- test never touches selection.")
print(f"Stratified by '{config['label_col']}', target_pos_frac={config['target_pos_frac']}")
print(f"Threshold: {config['threshold_method']} (={config['threshold']})" 
      if config['threshold_method'] == 'fixed' else f"Threshold: {config['threshold_method']}")
print(f"Warmup: {config['warmup_epochs']} epochs, patience: {config['patience']}, epoch_cap: {config['epoch_cap']}")
print(f"Model capacity: hidden_dim={model_cfg.get('hidden_dim')}, dropout={model_cfg.get('dropout')} "
      f"(matched to the 07 baseline, not 07b/07c's larger capacity)")
print(f"Full config: {config}")

In [ ]:
import pandas as pd
import graph_datasets as ds
import train as tr
import evaluate as ev
import models

index_df = pd.read_parquet(PROCESSED_DIR / "dataset_index.parquet")
dataset = ds.DualGraphDataset(index_df, PROCESSED_DIR / "svg_graphs", PROCESSED_DIR / "tvg_graphs")
print(f"Dataset: {len(dataset)} points (70:30 train/test branch -- no fold_cols, no bootstrap resampling)")

svg_kwargs = dict(hidden_dim=model_cfg.get("hidden_dim", 64), heads=model_cfg.get("heads", 4),
                   num_layers=model_cfg.get("svg_layers", 2), dropout=model_cfg.get("dropout", 0.35),
                   signage_vocab=5, light_pole_vocab=4, road_marking_vocab=2, cat_embed_dim=2)
tvg_kwargs = dict(hidden_dim=model_cfg.get("hidden_dim", 64), heads=model_cfg.get("heads", 4),
                   num_layers=model_cfg.get("tvg_layers", 2), dropout=model_cfg.get("dropout", 0.35),
                   building_type_vocab=58, highway_vocab=13,
                   building_type_embed_dim=8, highway_embed_dim=4)

In [ ]:
# ── Train every primary scenario x head depth ─────────────────────────
# Split into one cell per scenario below, same rationale as 07/07b/07c:
# run / monitor / interrupt independently rather than one long nested loop.
PRIMARY_SCENARIOS = ["A", "B", "C", "D", "E"]
HEAD_DEPTHS = ["linear", "mlp2"]
all_results = {}

### Scenario A — SVG only

In [ ]:
# ── Scenario A: linear + mlp2 ─────────────────────────────────
for depth in HEAD_DEPTHS:
    key = f"A_{depth}"
    print(f"\n=== {key} ===")
    results = tr.run_scenario_train_test_repeats("A", depth, use_ablation=False, dataset=dataset,
                                                   n_repeats=N_REPEATS, config=config, svg_kwargs=svg_kwargs,
                                                   tvg_kwargs=tvg_kwargs, device=device, checkpoint_dir=CHECKPOINT_DIR)
    all_results[key] = results
    print(f"  {len(results)} repeat-runs complete.")

### Scenario B — TVG only

In [ ]:
# ── Scenario B: linear + mlp2 ─────────────────────────────────
for depth in HEAD_DEPTHS:
    key = f"B_{depth}"
    print(f"\n=== {key} ===")
    results = tr.run_scenario_train_test_repeats("B", depth, use_ablation=False, dataset=dataset,
                                                   n_repeats=N_REPEATS, config=config, svg_kwargs=svg_kwargs,
                                                   tvg_kwargs=tvg_kwargs, device=device, checkpoint_dir=CHECKPOINT_DIR)
    all_results[key] = results
    print(f"  {len(results)} repeat-runs complete.")

### Scenario C — dual graph (concat)

In [ ]:
# ── Scenario C: linear + mlp2 ─────────────────────────────────
for depth in HEAD_DEPTHS:
    key = f"C_{depth}"
    print(f"\n=== {key} ===")
    results = tr.run_scenario_train_test_repeats("C", depth, use_ablation=False, dataset=dataset,
                                                   n_repeats=N_REPEATS, config=config, svg_kwargs=svg_kwargs,
                                                   tvg_kwargs=tvg_kwargs, device=device, checkpoint_dir=CHECKPOINT_DIR)
    all_results[key] = results
    print(f"  {len(results)} repeat-runs complete.")

### Scenario D — dual graph (late fusion)

In [ ]:
# ── Scenario D: linear + mlp2 ─────────────────────────────────
for depth in HEAD_DEPTHS:
    key = f"D_{depth}"
    print(f"\n=== {key} ===")
    results = tr.run_scenario_train_test_repeats("D", depth, use_ablation=False, dataset=dataset,
                                                   n_repeats=N_REPEATS, config=config, svg_kwargs=svg_kwargs,
                                                   tvg_kwargs=tvg_kwargs, device=device, checkpoint_dir=CHECKPOINT_DIR)
    all_results[key] = results
    print(f"  {len(results)} repeat-runs complete.")

### Scenario E — dual graph (cross-attention)

In [ ]:
# ── Scenario E: linear + mlp2 ─────────────────────────────────
for depth in HEAD_DEPTHS:
    key = f"E_{depth}"
    print(f"\n=== {key} ===")
    results = tr.run_scenario_train_test_repeats("E", depth, use_ablation=False, dataset=dataset,
                                                   n_repeats=N_REPEATS, config=config, svg_kwargs=svg_kwargs,
                                                   tvg_kwargs=tvg_kwargs, device=device, checkpoint_dir=CHECKPOINT_DIR)
    all_results[key] = results
    print(f"  {len(results)} repeat-runs complete.")

In [ ]:
# ── Ablation: B-E only (F deferred) ────────────────────────────────────
# Split into one cell per scenario, same rationale as the primary block above.

### Ablation B+

In [ ]:
# ── Scenario B ablation: linear + mlp2 ─────────────────────────────────
for depth in HEAD_DEPTHS:
    key = f"B_{depth}_ablation"
    print(f"\n=== {key} ===")
    results = tr.run_scenario_train_test_repeats("B", depth, use_ablation=True, dataset=dataset,
                                                   n_repeats=N_REPEATS, config=config, svg_kwargs=svg_kwargs,
                                                   tvg_kwargs=tvg_kwargs, device=device, checkpoint_dir=CHECKPOINT_DIR)
    all_results[key] = results
    print(f"  {len(results)} repeat-runs complete.")

### Ablation C+

In [ ]:
# ── Scenario C ablation: linear + mlp2 ─────────────────────────────────
for depth in HEAD_DEPTHS:
    key = f"C_{depth}_ablation"
    print(f"\n=== {key} ===")
    results = tr.run_scenario_train_test_repeats("C", depth, use_ablation=True, dataset=dataset,
                                                   n_repeats=N_REPEATS, config=config, svg_kwargs=svg_kwargs,
                                                   tvg_kwargs=tvg_kwargs, device=device, checkpoint_dir=CHECKPOINT_DIR)
    all_results[key] = results
    print(f"  {len(results)} repeat-runs complete.")

### Ablation D+

In [ ]:
# ── Scenario D ablation: linear + mlp2 ─────────────────────────────────
for depth in HEAD_DEPTHS:
    key = f"D_{depth}_ablation"
    print(f"\n=== {key} ===")
    results = tr.run_scenario_train_test_repeats("D", depth, use_ablation=True, dataset=dataset,
                                                   n_repeats=N_REPEATS, config=config, svg_kwargs=svg_kwargs,
                                                   tvg_kwargs=tvg_kwargs, device=device, checkpoint_dir=CHECKPOINT_DIR)
    all_results[key] = results
    print(f"  {len(results)} repeat-runs complete.")

### Ablation E+

In [ ]:
# ── Scenario E ablation: linear + mlp2 ─────────────────────────────────
for depth in HEAD_DEPTHS:
    key = f"E_{depth}_ablation"
    print(f"\n=== {key} ===")
    results = tr.run_scenario_train_test_repeats("E", depth, use_ablation=True, dataset=dataset,
                                                   n_repeats=N_REPEATS, config=config, svg_kwargs=svg_kwargs,
                                                   tvg_kwargs=tvg_kwargs, device=device, checkpoint_dir=CHECKPOINT_DIR)
    all_results[key] = results
    print(f"  {len(results)} repeat-runs complete.")

In [ ]:
# ── Scenario G: XGBoost, separate path (70:30 train/test version) ──────
import baseline_features
from xgboost import XGBClassifier

feat_table = baseline_features.build_feature_table(index_df["point_id"].tolist(),
                                                     PROCESSED_DIR / "svg_graphs", PROCESSED_DIR / "tvg_graphs", torch)
feat_table = feat_table.merge(index_df[["point_id", "label"]], on="point_id")
feature_cols = [c for c in feat_table.columns if c not in ["point_id", "label"]]

g_results = []
for repeat in range(N_REPEATS):
    repeat_seed = 42 + repeat
    # Same two-step split as run_scenario_train_test_repeats for A-E:
    # outer 70/30 (test held out genuinely), then the 70% split again
    # into train/val for threshold fitting only -- test is never reused
    # inside the inner split.
    train_full, _, test = tr._stratified_split(
        feat_table, config["label_col"], val_frac=0.0, test_frac=config["test_frac"],
        seed=repeat_seed, target_pos_frac=config.get("target_pos_frac"))
    train_, val_, _ = tr._stratified_split(
        train_full, config["label_col"], val_frac=config["internal_val_frac"], test_frac=0.0,
        seed=repeat_seed + 1, target_pos_frac=config.get("target_pos_frac"))

    clf = XGBClassifier(n_estimators=200, max_depth=4, eval_metric="aucpr", random_state=42)
    clf.fit(train_[feature_cols], train_["label"])

    if config["threshold_method"] == "fixed":
        chosen_threshold = config.get("threshold", 0.5)
        threshold_method_used = "fixed"
    else:
        val_prob = clf.predict_proba(val_[feature_cols])[:, 1]
        chosen_threshold, threshold_method_used, _ = ev.find_optimal_threshold(
            val_["label"].values, val_prob, method=config["threshold_method"])

    prob = clf.predict_proba(test[feature_cols])[:, 1]
    metrics = ev.compute_metrics(test["label"].values, prob, threshold=chosen_threshold)
    metrics["threshold_method"] = threshold_method_used
    g_results.append({"repeat": repeat, "n_train": len(train_), "n_val": len(val_), "n_test": len(test),
                       **metrics})

all_results["G"] = g_results
print(f"Scenario G: {len(g_results)} repeat-runs complete.")

In [ ]:
# ── Aggregate + report EVERY scenario, regardless of performance ────────
agg_rows = []
for key, results in all_results.items():
    agg = ev.aggregate_fold_results(results)   # works identically regardless of split scheme
    row = {"scenario": key}
    for metric, (mean, std) in agg.items():
        row[f"{metric}_mean"] = mean
        row[f"{metric}_std"] = std
    agg_rows.append(row)

agg_df = pd.DataFrame(agg_rows)
agg_df.to_csv(METRICS_DIR / "all_scenarios_summary_70_30.csv", index=False)
display(agg_df)

## Threshold diagnostics

Threshold is fixed at 0.5 for every repeat in this branch, so
`threshold_used` should be constant and `threshold_method` should read
`"fixed"` for all rows -- this cell mainly exists as a sanity check that
nothing accidentally fell back to a different method.

In [ ]:
threshold_rows = []
for key, results in all_results.items():
    method_counts = ev.summarize_categorical_field(results, "threshold_method")
    thresh_mean, thresh_std = ev.aggregate_fold_results(results).get("threshold_used", (float("nan"), float("nan")))
    threshold_rows.append({"scenario": key, "threshold_mean": thresh_mean, "threshold_std": thresh_std,
                            "methods_used": method_counts})

threshold_df = pd.DataFrame(threshold_rows)
threshold_df.to_csv(METRICS_DIR / "threshold_diagnostics_70_30.csv", index=False)
display(threshold_df)

In [ ]:
# ── Head-depth comparison: descriptive only (no formal significance test
#    here -- see notebook intro for why). ──
depth_compare = []
for scenario in PRIMARY_SCENARIOS:
    lin = agg_df[agg_df["scenario"] == f"{scenario}_linear"]["pr_auc_mean"].iloc[0]
    mlp = agg_df[agg_df["scenario"] == f"{scenario}_mlp2"]["pr_auc_mean"].iloc[0]
    depth_compare.append({"scenario": scenario, "linear_pr_auc": lin, "mlp2_pr_auc": mlp})

depth_df = pd.DataFrame(depth_compare)
display(depth_df)

WINNING_DEPTH = "linear" if depth_df["linear_pr_auc"].mean() >= depth_df["mlp2_pr_auc"].mean() else "mlp2"
print(f"\nWinning head depth (by mean PR-AUC across scenarios): {WINNING_DEPTH}")
print("Descriptive only -- no Wilcoxon/Nadeau-Bengio run in this branch (see intro).")

## Comparing against 07 (formal k-fold baseline)

**Note:** 07 uses spatial k-fold (`configs/eval.yaml`) with `hidden_dim=64`
(matched here on purpose); this branch instead uses repeated 70:30
train/test re-splits. So this isn't a controlled ablation of the split
scheme alone in the strictest sense (k-fold vs. random-repeat are
different partition structures, not just different ratios) -- but since
model capacity is held constant against 07 specifically, the comparison
is at least not confounded by capacity, unlike a comparison against
07b/07c would be.

Loads 07's `all_scenarios_summary.csv` (if you've already generated one
there under an equivalent name) alongside this branch's summary and
compares mean PR-AUC + accuracy for matching scenario/depth keys.

In [ ]:
baseline_summary_path = OUTPUTS_DIR / "metrics" / "all_scenarios_summary.csv"
if baseline_summary_path.exists():
    base_df = pd.read_csv(baseline_summary_path)
    cols = ["scenario", "pr_auc_mean", "pr_auc_std"]
    if "accuracy_mean" in base_df.columns:
        cols += ["accuracy_mean"]
    compare_cols = [c for c in cols if c in agg_df.columns and c in base_df.columns] + ["scenario"]
    compare_cols = list(dict.fromkeys(compare_cols))
    compare = agg_df[compare_cols].merge(
        base_df[compare_cols], on="scenario", suffixes=("_70_30", "_kfold_baseline"))
    display(compare)
else:
    print("07's summary CSV not found at the expected path -- adjust "
          "baseline_summary_path above, or run/export 07 first to compare.")

## Epoch-level diagnostics

Per-repeat training history is saved under
`CHECKPOINT_DIR/{tag}_history/repeat{N}.json`, same JSON shape as
07b/07c. `plot_history.py`'s existing helpers assume `{fold_col}_fold{id}`
naming, not `repeat{N}` -- using the same direct JSON-loading fallback as
07b/07c rather than assuming an unverified helper exists.

In [ ]:
import json
import matplotlib.pyplot as plt

history_path = CHECKPOINT_DIR / "A_linear_history" / "repeat0.json"
history = json.loads(history_path.read_text())

epochs = [h["epoch"] for h in history]
best_epoch = max(range(len(history)), key=lambda i: history[i]["val_pr_auc"])

fig, ax1 = plt.subplots(figsize=(8, 4))
ax1.plot(epochs, [h["train_loss"] for h in history], label="train_loss", color="tab:blue")
ax1.set_xlabel("epoch"); ax1.set_ylabel("train_loss", color="tab:blue")

ax2 = ax1.twinx()
ax2.plot(epochs, [h["val_pr_auc"] for h in history], label="val_pr_auc", color="tab:orange")
ax2.plot(epochs, [h["val_auroc"] for h in history], label="val_auroc", color="tab:green")
ax2.axvline(best_epoch, color="gray", linestyle="--", label=f"best epoch ({best_epoch})")
ax2.set_ylabel("val metric")

lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, fontsize=8, loc="best")
plt.title("A_linear, repeat 0 -- 70:30 branch")
plt.tight_layout()
plt.show()

In [ ]:
print("70:30 train/test branch complete.")
print("Compare mean/std + accuracy against 07 (k-fold baseline, same capacity)")
print("to see whether the split ratio itself (vs. k-fold partitioning) changes the picture.")